# gkmsvm-lite application note figures

Reads result files from `results/` and writes figures to `figures/`. Nothing here
computes results or needs a GPU. Each input comes from one `make` target in
`benchmarks/Makefile`; see `benchmarks/README.md`.

| Panel | Input (`results/`) | Make target |
|---|---|---|
| 1A | `agreement_dsqtl.tsv` | `fig1a` |
| 1B | `timing_train.tsv`, `timing_train_T1.tsv` | `fig1b` |
| 1C | `timing_variants.tsv` | `fig1c` |
| 1D | `dsqtl_summed.tsv` | `fig1d` |
| S1 | `dsqtl_retrained_per_model.tsv`, `dsqtl_per_model.tsv` | `s1`, `fig1d` |
| S2 | `throughput.tsv` | `s2` |
| S3 | `training_memory.tsv`, `training_memory_cpu.tsv` | `s3` |
| S4 | `completeness.tsv` | `s4` |
| S5 | `dart_task4_*.tsv`, `dart_task5_dsqtl_*.tsv` | `s5` |

A panel whose input is missing is drawn as a labeled empty box, never with
placeholder numbers. The last cell lists missing inputs; set `STRICT=1` to make
it raise. `make figures QUICK=1` reads `results/quick/` and writes `figures/quick/`.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from _common import read_provenance  # noqa: E402

RESULTS = Path(os.environ.get("RESULTS_DIR", "results"))
FIGURES = Path(os.environ.get("FIGURES_DIR", "figures"))
FIGURES.mkdir(parents=True, exist_ok=True)

# Bioinformatics style: 178 mm double column
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 7, "axes.titlesize": 8, "axes.labelsize": 7,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 6,
    "figure.dpi": 150, "savefig.dpi": 300, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42,
})
FIG_W = 7.0

C_LSGKM = "#7d7f85"   # LS-GKM C
C_LSGKM1 = "#b4b6bb"  # LS-GKM C, 1 thread
C_MLSGKM = "#8c6d31"  # mLS-GKM (multithreaded LS-GKM fork)
C_MLSGKM1 = "#d2bd8a"  # mLS-GKM, 1 thread
C_CPU = "#4682B4"     # gkmsvm-lite CPU
C_CPU1 = "#a9c6de"    # gkmsvm-lite CPU, 1 thread
C_GPU = "#D64848"     # gkmsvm-lite CUDA
C_DSVM = "#5A9418"    # deltaSVM
C_ISM = "#C99400"     # ISM
C_EXPL = "#6C66B0"    # GkmExplain mutation impact
METHOD_COLORS = {"deltaSVM": C_DSVM, "deltaSVM.pl": C_LSGKM, "ISM": C_ISM, "GkmExplain": C_EXPL}

MISSING = []


def load(name):
    """Read results/<name> (TSV with a provenance header), or None if absent."""
    path = RESULTS / name
    if not path.exists():
        MISSING.append(name)
        print(f"MISSING {path}")
        return None
    prov = read_provenance(path)
    print(f"{name}: commit {prov.get('commit')}, {prov.get('host')}, "
          f"{prov.get('gpu') or prov.get('device', '')}")
    return pd.read_csv(path, sep="\t", comment="#")


def missing_panel(ax, name):
    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(True)
        s.set_linestyle(":")
        s.set_color("0.6")
    ax.text(0.5, 0.5, f"missing\nresults/{name}", ha="center", va="center",
            color="0.45", transform=ax.transAxes)


def letter(ax, s, x=-0.18):
    ax.text(x, 1.06, s, transform=ax.transAxes, fontsize=10, fontweight="bold", va="bottom")


def save(fig, stem):
    for ext in ("pdf", "png"):
        fig.savefig(FIGURES / f"{stem}.{ext}")
    plt.show()


def load_timing(name):
    """Parse the two-section TSV written by timing_train.py."""
    path = RESULTS / name
    if not path.exists():
        MISSING.append(name)
        print(f"MISSING {path}")
        return None, None, None
    prov = read_provenance(path)
    print(f"{name}: commit {prov.get('commit')}, {prov.get('gpu', prov.get('device'))}, "
          f"LS-GKM -T {prov.get('lsgkm_threads')}, "
          f"gkmsvm-lite CPU threads {prov.get('lite_cpu_threads')}")
    sections, current = {}, None
    for line in path.read_text().splitlines():
        if line.startswith("## "):
            current = line[3:].strip().lower()
            sections[current] = []
        elif current and line and not line.startswith("#"):
            sections[current].append(line.split("\t"))
    frames = {}
    for key, rows in sections.items():
        df = pd.DataFrame(rows[1:], columns=rows[0]).replace("NA", np.nan)
        frames[key] = df.apply(pd.to_numeric)
    # Both implementations run CPU code on this many threads
    threads = prov.get("lite_cpu_threads") or prov.get("lsgkm_threads")
    return frames.get("training"), frames.get("inference"), threads

## Figure 1

In [ ]:
agree = load("agreement_dsqtl.tsv")
train, infer, n_threads = load_timing("timing_train.tsv")
train_t1, _, _ = load_timing("timing_train_T1.tsv")
variants = load("timing_variants.tsv")
summed = load("dsqtl_summed.tsv")

In [ ]:
fig = plt.figure(figsize=(FIG_W, 4.6))
outer = fig.add_gridspec(2, 2, hspace=0.55, wspace=0.32)

# A: per-variant agreement with LS-GKM, first -t 3 model in the dump
gA = outer[0, 0].subgridspec(1, 3, wspace=0.55)
kinds = [("ISM", "ISM"), ("GkmExplain", "GkmExplain"), ("deltaSVM.pl", "deltaSVM")]
axA = [fig.add_subplot(gA[0, i]) for i in range(3)]
letter(axA[0], "A", x=-0.6)
if agree is None:
    for ax in axA:
        missing_panel(ax, "agreement_dsqtl.tsv")
else:
    prefix = sorted({c.rsplit("_", 2)[0] for c in agree.columns if c.startswith("t3_")})[0]
    for ax, (kind, title) in zip(axA, kinds):
        x = agree[f"{prefix}_{kind}_lsgkm"].to_numpy()
        y = agree[f"{prefix}_{kind}_lite"].to_numpy()
        ax.scatter(x, y, s=1, color=METHOD_COLORS[kind], alpha=0.5, lw=0, rasterized=True)
        lim = np.abs(np.concatenate([x, y])).max() * 1.05
        ax.plot([-lim, lim], [-lim, lim], color="0.3", lw=0.5, ls="--")
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_aspect("equal")
        ax.set_title(title, pad=3)
        ax.text(0.04, 0.96, f"max |Δ|\n{np.abs(y - x).max():.0e}", transform=ax.transAxes,
                va="top", fontsize=5.5)
        ax.set_xlabel("LS-GKM")
    axA[0].set_ylabel("gkmsvm-lite")

# B: training time vs N
axB = fig.add_subplot(outer[0, 1])
letter(axB, "B")
if train is None:
    missing_panel(axB, "timing_train.tsv")
else:
    # Solid: both at n_threads; dashed: both at 1 thread
    axB.plot(train["N"], train["t_lsgkm_train"], "o-", color=C_LSGKM, ms=3, lw=1,
             label=f"LS-GKM, {n_threads} threads")
    axB.plot(train["N"], train["t_lite_cpu_train"], "s-", color=C_CPU, ms=3, lw=1,
             label=f"gkmsvm-lite CPU, {n_threads} threads")
    if train_t1 is not None:
        axB.plot(train_t1["N"], train_t1["t_lsgkm_train"], "o--", color=C_LSGKM1, ms=3,
                 lw=1, label="LS-GKM, 1 thread")
        axB.plot(train_t1["N"], train_t1["t_lite_cpu_train"], "s--", color=C_CPU1, ms=3,
                 lw=1, label="gkmsvm-lite CPU, 1 thread")
    if train["t_lite_gpu_train"].notna().any():
        axB.plot(train["N"], train["t_lite_gpu_train"], "^-", color=C_GPU, ms=3, lw=1,
                 label="gkmsvm-lite GPU")
    axB.set_xscale("log")
    axB.set_yscale("log")
    axB.set_xlabel("Training sequences")
    axB.set_ylabel("Training time (s)")
    axB.legend(frameon=False, loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2,
               fontsize=6, handlelength=2.2, columnspacing=1.0, borderaxespad=0.2)
    axB.grid(True, which="major", alpha=0.2)

# C: time to score the dsQTL variants. ISM and GkmExplain score each variant
# against the SVs; deltaSVM builds a 10-mer table once per model, then scores
# variants with table lookups
axC = fig.add_subplot(outer[1, 0])
letter(axC, "C")
if variants is None:
    missing_panel(axC, "timing_variants.tsv")
else:
    v = variants[variants["status"] == "ok"]
    order = ["ISM", "GkmExplain", "deltaSVM", "deltaSVM scoring"]
    ylabels = ["ISM", "GkmExplain", "deltaSVM:\nbuild table (once)", "deltaSVM:\nscore variants"]
    cpu = v[v["device"] == "cpu"]
    t_max = int(cpu["threads"].max()) if len(cpu) else None
    # (implementation, device, CPU threads or None for any, color, label), CPU
    # bars in pairs at the same thread count
    c_impls = [(i, c, c1) for i, c, c1 in (("LS-GKM", C_LSGKM, C_LSGKM1),
                                           ("mLS-GKM", C_MLSGKM, C_MLSGKM1))
               if (v["implementation"] == i).any()]
    impls = []
    for thr in dict.fromkeys([t_max, 1]):
        many = thr > 1
        impls += [(i, "cpu", thr, c if many else c1, f"{i}, {thr} thread{'s' if many else ''}")
                  for i, c, c1 in c_impls]
        impls.append(("gkmsvm-lite", "cpu", thr, C_CPU if many else C_CPU1,
                      f"gkmsvm-lite CPU, {thr} thread{'s' if many else ''}"))
    impls.append(("gkmsvm-lite", "cuda", None, C_GPU, "gkmsvm-lite GPU"))
    h = 0.8 / len(impls)

    def fmt(t):
        return f"{t / 60:.1f} min" if t >= 60 else f"{t:.0f} s" if t >= 10 else f"{t:.1f} s"

    for j, (impl, dev, thr, color, label) in enumerate(impls):
        sub = v[(v["implementation"] == impl) & (v["device"] == dev)]
        if thr is not None:
            sub = sub[sub["threads"] == thr]
        sub = sub.set_index("method")
        if sub.empty:
            continue
        labeled = False
        for i, m in enumerate(order):
            if m not in sub.index:
                continue
            t = sub.loc[m, "wall_s"]
            y = i + (j - (len(impls) - 1) / 2) * h
            axC.barh(y, t, height=h, color=color, label=None if labeled else label)
            labeled = True
            txt = f" {fmt(t)}"
            if m == "deltaSVM scoring":  # wall time is process start-up and JIT
                txt += f" ({sub.loc[m, 'compute_s'] * 1e3:.0f} ms scoring)"
            axC.text(t, y, txt, va="center", fontsize=4.5)
    axC.axhline(1.5, color="0.6", lw=0.5, ls=":")
    axC.set_xscale("log")
    axC.set_yticks(range(len(order)), ylabels)
    axC.invert_yaxis()
    n_var = int(v.loc[v["method"] != "deltaSVM", "n_items"].max())
    axC.set_xlabel(f"Wall-clock time (s), {n_var:,} dsQTL variants")
    # Below the axis: up to 7 entries are wider than the panel
    axC.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.45, -0.25),
               handlelength=1.0, columnspacing=1.0, fontsize=5.5)

# D: summed dsQTL AUPRC, both kernels
axD = fig.add_subplot(outer[1, 1])
letter(axD, "D")
if summed is None:
    missing_panel(axD, "dsqtl_summed.tsv")
else:
    summed["base"] = summed["method"].str.replace(r"-sum.*", "", regex=True)
    order = ["deltaSVM", "deltaSVM.pl", "ISM", "GkmExplain"]
    for params, marker, fill in (("paper_t3", "o", True), ("paper_t2", "D", False)):
        sub = summed[summed["params"] == params].set_index("base")
        if sub.empty:
            continue
        for i, m in enumerate(order):
            if m not in sub.index:
                continue
            c = METHOD_COLORS[m]
            axD.scatter(sub.loc[m, "auprc"], i, marker=marker, s=22, zorder=3,
                        facecolor=c if fill else "white", edgecolor=c, lw=1.2)
            ref = sub.loc[m, "paper_auprc"]
            if pd.notna(ref):
                axD.scatter(ref, i, marker="|", s=60, color="0.2", lw=0.8, zorder=4)
    axD.set_yticks(range(len(order)))
    axD.set_yticklabels(order)
    axD.invert_yaxis()
    axD.set_xlabel("dsQTL AUPRC (5 models summed)")
    from matplotlib.lines import Line2D
    handles = [
        Line2D([], [], marker="o", ls="", color="0.35", ms=4, label="-t 3 gkmrbf"),
        Line2D([], [], marker="D", ls="", mfc="white", mec="0.35", ms=3.5, label="-t 2 gkm_esttrunc"),
        Line2D([], [], marker="|", ls="", color="0.2", ms=7, label="published"),
    ]
    axD.legend(handles=handles, frameon=False, ncol=3, loc="lower center",
               bbox_to_anchor=(0.5, 1.0), handletextpad=0.2, columnspacing=1.0)
    axD.grid(True, axis="x", alpha=0.2)

save(fig, "fig1")

In [ ]:
# Numbers quoted in the text
rows = []
if train is not None and train["t_lite_gpu_train"].notna().any():
    r = train.dropna(subset=["t_lsgkm_train", "t_lite_gpu_train"]).iloc[-1]
    rows.append({"quantity": f"training speedup, GPU vs LS-GKM {n_threads} threads, "
                             f"N={int(r['N'])}",
                 "value": r["t_lsgkm_train"] / r["t_lite_gpu_train"]})
# CPU speedups compare runs from the same file, so both sides use the same threads
for df, t in ((train, n_threads), (train_t1, 1)):
    if df is None:
        continue
    r = df.dropna(subset=["t_lsgkm_train", "t_lite_cpu_train"]).iloc[-1]
    rows.append({"quantity": f"training speedup, CPU vs LS-GKM, both {t} thread(s), "
                             f"N={int(r['N'])}",
                 "value": r["t_lsgkm_train"] / r["t_lite_cpu_train"]})
if infer is not None and infer["t_lite_gpu_infer"].notna().any():
    r = infer.dropna(subset=["t_lsgkm_infer", "t_lite_gpu_infer"]).iloc[-1]
    rows.append({"quantity": f"prediction speedup, GPU vs LS-GKM, {int(r['n_sv'])} SVs",
                 "value": r["t_lsgkm_infer"] / r["t_lite_gpu_infer"]})
if variants is not None:
    v = variants[variants["status"] == "ok"]
    for m in v["method"].unique():
        lite = v[(v["method"] == m) & (v["implementation"] == "gkmsvm-lite")]
        name = "deltaSVM table build" if m == "deltaSVM" else m
        for impl in ("LS-GKM", "mLS-GKM"):
            base = v[(v["method"] == m) & (v["implementation"] == impl)]
            if base.empty:
                continue
            # CPU against the C implementation at the same thread count
            for _, b in base.iterrows():
                cpu = lite[(lite["device"] == "cpu") & (lite["threads"] == b["threads"])]
                if len(cpu):
                    rows.append({"quantity": f"{name} speedup, gkmsvm-lite CPU vs {impl}, "
                                             f"both {int(b['threads'])} thread(s)",
                                 "value": b["wall_s"] / cpu["wall_s"].iloc[0]})
            # GPU against its fastest run
            fastest = base.loc[base["wall_s"].idxmin()]
            gpu = lite[lite["device"] == "cuda"]
            if len(gpu):
                rows.append({"quantity": f"{name} speedup, gkmsvm-lite GPU vs fastest {impl} "
                                         f"({int(fastest['threads'])} thread(s))",
                             "value": fastest["wall_s"] / gpu["wall_s"].iloc[0]})
    mem_cols = ["method", "implementation", "device", "threads", "peak_rss_mb", "gpu_pool_mb"]
    variants[mem_cols].to_csv(FIGURES / "table_variant_memory.tsv", sep="\t", index=False)
if summed is not None:
    for _, r in summed.iterrows():
        rows.append({"quantity": f"dsQTL AUPRC {r['method']}", "value": r["auprc"]})
numbers = pd.DataFrame(rows)
numbers.to_csv(FIGURES / "text_numbers.tsv", sep="\t", index=False)
numbers

## Supplement

In [ ]:
# S1: gkmsvm-lite-trained models vs the paper's lsgkm models, per negset
retrained = load("dsqtl_retrained_per_model.tsv")
paper = load("dsqtl_per_model.tsv")

fig, ax = plt.subplots(figsize=(3.2, 3.0))
if retrained is None or paper is None:
    missing_panel(ax, "dsqtl_retrained_per_model.tsv")
else:
    pairs = {"l10k6": "paper_t2", "l10k6_rbf": "paper_t3"}
    merged = retrained.assign(paper_params=retrained["params"].map(pairs)).merge(
        paper, left_on=["paper_params", "negset", "method"],
        right_on=["params", "negset", "method"], suffixes=("_lite", "_lsgkm"))
    for (method, params), g in merged.groupby(["method", "params_lite"]):
        ax.scatter(g["auprc_lsgkm"], g["auprc_lite"], s=14, color=METHOD_COLORS[method],
                   marker="o" if params.endswith("rbf") else "D",
                   label=f"{method}, {'-t 3' if params.endswith('rbf') else '-t 2'}")
    lo = merged[["auprc_lite", "auprc_lsgkm"]].min().min() - 0.002
    hi = merged[["auprc_lite", "auprc_lsgkm"]].max().max() + 0.002
    ax.plot([lo, hi], [lo, hi], color="0.3", lw=0.5, ls="--")
    ax.set_xlabel("AUPRC, LS-GKM-trained model")
    ax.set_ylabel("AUPRC, gkmsvm-lite-trained model")
    ax.legend(frameon=False, fontsize=5)
save(fig, "figS1_retrained")

In [ ]:
# S2: prediction throughput vs number of SVs
tput = load("throughput.tsv")
fig, axes = plt.subplots(1, 2, figsize=(FIG_W, 2.5), sharey=True)
TITLES = {"t2": "-t 2 (exact l-mer table)", "t3": "-t 3 (l-mer index)"}
for ax, kernel in zip(axes, ("t2", "t3")):
    ax.set_title(TITLES[kernel])
    if tput is None:
        missing_panel(ax, "throughput.tsv")
        continue
    t = tput[(tput["kernel"] == kernel) & (tput["status"] == "ok")]
    for (impl, path, dev), g in t.groupby(["implementation", "path", "device"]):
        g = g.sort_values("n_sv")
        if impl in ("LS-GKM", "mLS-GKM"):
            color, style = (C_LSGKM, "D-") if impl == "LS-GKM" else (C_MLSGKM, "s-")
            label = f"{impl} gkmpredict, {int(g['threads'].iloc[0])} threads"
        else:
            color = C_GPU if dev == "cuda" else C_CPU
            style = "o--" if path == "masked join" else "o-"
            label = f"gkmsvm-lite {'GPU' if dev == 'cuda' else 'CPU'}, {path}"
        ax.plot(g["n_sv"], g["seqs_per_sec"], style, color=color, ms=3, lw=1, label=label,
                mfc="none" if path == "masked join" else color)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Support vectors")
    ax.grid(True, which="major", alpha=0.2)
    ax.legend(frameon=False, fontsize=5.5)
axes[0].set_ylabel("Sequences per second (300 bp)")
letter(axes[0], "A")
letter(axes[1], "B")
save(fig, "figS2_throughput")

In [ ]:
# S3: training time and peak memory by solver
memory = load("training_memory.tsv")
memory_cpu = load("training_memory_cpu.tsv")
if memory is not None and memory_cpu is not None:
    memory = pd.concat([memory, memory_cpu], ignore_index=True)
fig, axes = plt.subplots(1, 2, figsize=(FIG_W, 2.5))
SOLVER_STYLE = {
    "libsvm": ("precomputed Gram", C_GPU, "o-"),
    "smo": ("SMO", C_CPU, "s-"),
    "nystrom": ("Nyström", C_EXPL, "^--"),
    "lsgkm": ("LS-GKM", C_LSGKM, "D-"),
}
if memory is None:
    for ax in axes:
        missing_panel(ax, "training_memory.tsv")
else:
    ok = memory[memory["status"] == "ok"]
    for (solver, device), g in ok.groupby(["solver", "device"]):
        if solver not in SOLVER_STYLE:
            continue
        label, color, style = SOLVER_STYLE[solver]
        g = g.sort_values("N")
        # gkmsvm-lite runs on the GPU are filled, on CPU open; LS-GKM is CPU only
        cpu = device == "cpu" and solver != "lsgkm"
        if solver != "lsgkm":
            label = f"{label}, {'CPU' if cpu else 'GPU'}"
        kw = dict(color=color, ms=3, lw=1, label=label, mfc="none" if cpu else color)
        if cpu:
            style = style[0] + ":"
        axes[0].plot(g["N"], g["train_s"], style, **kw)
        axes[1].plot(g["N"], g["peak_rss_mb"] / 1024, style, **kw)
    gram = memory.drop_duplicates("N").sort_values("N")
    axes[1].plot(gram["N"], gram["gram_gb"], color="0.6", lw=0.8, ls=":",
                 label="N² × 8 bytes")
    for ax, ylabel in zip(axes, ("Training time (s)", "Peak host memory (GB)")):
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("Training sequences")
        ax.set_ylabel(ylabel)
        ax.grid(True, which="major", alpha=0.2)
    axes[1].legend(frameon=False, fontsize=5.5)
    nys = ok[ok["solver"] == "nystrom"].sort_values("N")
    if len(nys) and nys["r_vs_exact"].notna().any():
        last = nys.dropna(subset=["r_vs_exact"]).iloc[-1]
        axes[0].text(0.03, 0.97, f"Nyström vs exact at N={int(last['N']):,}:\n"
                     f"r = {last['r_vs_exact']:.4f}, AUROC {last['test_auroc']:.4f}",
                     transform=axes[0].transAxes, va="top", fontsize=5.5)
letter(axes[0], "A")
letter(axes[1], "B")
save(fig, "figS3_solvers")

In [ ]:
# S4: GkmExplain completeness, sum of attributions = f(x) - b
compl = load("completeness.tsv")
fig, ax = plt.subplots(figsize=(2.8, 2.6))
if compl is None:
    missing_panel(ax, "completeness.tsv")
else:
    colors = {"positive": C_GPU, "negative": C_CPU, "random": "0.6"}
    for group, g in compl.groupby("group"):
        ax.scatter(g["score_minus_bias"], g["attribution_sum"], s=6, lw=0,
                   color=colors.get(group, "0.3"), label=group)
    x = compl["score_minus_bias"]
    ax.plot([x.min(), x.max()], [x.min(), x.max()], color="0.3", lw=0.5, ls="--")
    err = (compl["attribution_sum"] - x).abs().max()
    ax.text(0.04, 0.96, f"max |Δ| = {err:.1e}", transform=ax.transAxes, va="top")
    ax.set_xlabel("f(x) − b")
    ax.set_ylabel("Σ GkmExplain attributions")
    ax.legend(frameon=False, loc="lower right")
save(fig, "figS4_completeness")

In [ ]:
# S5: DART-Eval (Pampari et al. 2024), GM12878 ENCODE models
# Published reference values: Task 4 Table 2 (mean AUROC), Task 5 Fig. 6 (dsQTL AUPRC)
DART_REF = {
    "task4": {"Probed DNABERT-2": 0.784, "Probed NT-v2": 0.791, "Probed HyenaDNA": 0.800,
              "Fine-tuned NT-v2": 0.935, "ChromBPNet": 0.946},
    "task5": {"Enformer": 0.33, "ChromBPNet": 0.43},
}


def load_glob(pattern):
    files = sorted(RESULTS.glob(pattern))
    if not files:
        MISSING.append(pattern)
        print(f"MISSING {RESULTS / pattern}")
        return None
    return pd.concat([pd.read_csv(f, sep="\t", comment="#").assign(model=f.stem.rsplit("_", 1)[-1])
                      for f in files], ignore_index=True)


task4 = load_glob("dart_task4_*.tsv")
task5 = load_glob("dart_task5_dsqtl_*.tsv")

fig, axes = plt.subplots(1, 2, figsize=(FIG_W, 2.4))
for ax, df, key, metric, name in ((axes[0], task4, "task4", "auroc", "dart_task4_*.tsv"),
                                   (axes[1], task5, "task5", "auprc", "dart_task5_dsqtl_*.tsv")):
    if df is None:
        missing_panel(ax, name)
        continue
    labels = [f"gkmsvm-lite {m}" for m in df["model"]] + list(DART_REF[key])
    values = list(df[metric]) + list(DART_REF[key].values())
    colors = [C_CPU] * len(df) + ["0.75"] * len(DART_REF[key])
    ax.barh(range(len(values)), values, color=colors, height=0.6)
    ax.set_yticks(range(len(values)), labels)
    ax.invert_yaxis()
    ax.set_xlabel(metric.upper())
    for i, v in enumerate(values):
        ax.text(v, i, f" {v:.3f}", va="center", fontsize=5.5)
axes[0].set_title("Task 4: chromatin activity")
axes[1].set_title("Task 5: dsQTL variant effects")
save(fig, "figS5_dart_eval")

In [ ]:
if MISSING:
    print("Missing inputs:\n  " + "\n  ".join(MISSING))
    if os.environ.get("STRICT") == "1":
        raise FileNotFoundError(f"{len(MISSING)} missing result files")
else:
    print("All inputs present.")